# Understanding Context Engineering: Stateless vs. Stateful Agents

This notebook accompanies the article "Understanding Context Engineering: Why Your AI Agent Keeps Forgetting (And How to Fix It)"

You'll learn:
1. Why LLMs are inherently stateless
2. **Context Engineering as a systematic discipline** *(New)*
3. **The four layers of context** *(New)*
4. How Google ADK Sessions solve the stateless problem
5. **Memory taxonomy: semantic, episodic, procedural** *(New)*
6. How Sessions, State, and Memory work together

---

## Setup

First, let's install and import the required packages.

In [1]:
# Install required packages if not already installed
!pip install google-adk python-dotenv -q


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
from pathlib import Path
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load environment variables from .env file
# This looks for .env in the project root (parent of notebooks/)
env_path = Path(__file__).parent.parent / ".env" if "__file__" in globals() else Path("../.env")
load_dotenv(dotenv_path=env_path)

# Get API key from environment
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found! Please:\n"
        "1. Copy .env.example to .env\n"
        "2. Add your API key to .env\n"
        "3. Get your key from: https://aistudio.google.com/apikey"
    )

os.environ["GOOGLE_API_KEY"] = api_key

# Initialize the client
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"

print("✅ Environment loaded successfully")
print(f"🔑 API key loaded: {api_key[:2]}...{api_key[-2:]}")  # Show only first 8 and last 4 chars

✅ Environment loaded successfully
🔑 API key loaded: AI...go


## Part 1: The Stateless Problem

Let's demonstrate why LLMs "forget" between calls. Each API call is completely independent—the model has no memory of previous interactions.

In [2]:
def stateless_call(message: str) -> str:
    """Make a stateless API call - no conversation history."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=message
    )
    return response.text

In [5]:
# First message - introduce ourselves
response1 = stateless_call("My name is Alex and I'm a software engineer who loves hiking.")
print("User: My name is Alex and I'm a software engineer who loves hiking.")
print(f"Agent: {response1}")

User: My name is Alex and I'm a software engineer who loves hiking.
Agent: Hi Alex! That's a great combination. Sounds like you get a good balance between the digital world and the great outdoors.

It's common for software engineers to love hiking – a perfect way to clear the head and escape the screens!

What kind of hiking do you usually enjoy? Or what's your favorite part about hitting the trails?


In [6]:
# Second message - ask about what we just said
response2 = stateless_call("What's my name and what do I do for work?")
print("User: What's my name and what do I do for work?")
print(f"Agent: {response2}")

User: What's my name and what do I do for work?
Agent: As an AI, I don't know your name or what you do for work. I don't have access to personal information about you.

If you'd like to tell me, I'm happy to know! Otherwise, feel free to ask me anything else.


### 🔴 The Problem

The agent has no idea who you are! Each API call is independent—the model doesn't retain any information from the previous call.

This is the **stateless problem**: LLMs don't have built-in memory between requests.

---
## Part 2: Context Engineering as a Discipline

Context Engineering isn't just "passing data to an LLM"—it's a systematic discipline of designing the complete informational environment for AI models.

> **Context Engineering reframes the task from merely answering a question to building a comprehensive operational picture for the agent.**
> — *Agentic Design Patterns*, Antonio Gulli (2025)

In [7]:
# Let's visualize the four layers of context

context_layers = {
    "Layer 1: System Prompt": {
        "description": "Foundational instructions defining operational parameters",
        "example": "You are a travel agent; be concise and helpful",
        "persistence": "Static per agent"
    },
    "Layer 2: Retrieved Documents": {
        "description": "Information actively fetched from knowledge bases",
        "example": "Technical specifications, user manuals, product docs",
        "persistence": "Per query (RAG)"
    },
    "Layer 3: Tool Outputs": {
        "description": "Results from external API calls",
        "example": "Calendar availability, weather data, search results",
        "persistence": "Per invocation"
    },
    "Layer 4: Conversation History": {
        "description": "Prior exchanges in the current session",
        "example": "User messages and agent responses",
        "persistence": "Session duration"
    },
    "Layer 5: Implicit Data": {
        "description": "User identity, interaction patterns, environmental state",
        "example": "User preferences, time zone, device type",
        "persistence": "Long-term (Memory)"
    }
}

print("THE FOUR LAYERS OF CONTEXT ENGINEERING")
print("=" * 60)
for layer, details in context_layers.items():
    print(f"\n{layer}")
    print(f"  📝 {details['description']}")
    print(f"  💡 Example: {details['example']}")
    print(f"  ⏱️  Persistence: {details['persistence']}")

THE FOUR LAYERS OF CONTEXT ENGINEERING

Layer 1: System Prompt
  📝 Foundational instructions defining operational parameters
  💡 Example: You are a travel agent; be concise and helpful
  ⏱️  Persistence: Static per agent

Layer 2: Retrieved Documents
  📝 Information actively fetched from knowledge bases
  💡 Example: Technical specifications, user manuals, product docs
  ⏱️  Persistence: Per query (RAG)

Layer 3: Tool Outputs
  📝 Results from external API calls
  💡 Example: Calendar availability, weather data, search results
  ⏱️  Persistence: Per invocation

Layer 4: Conversation History
  📝 Prior exchanges in the current session
  💡 Example: User messages and agent responses
  ⏱️  Persistence: Session duration

Layer 5: Implicit Data
  📝 User identity, interaction patterns, environmental state
  💡 Example: User preferences, time zone, device type
  ⏱️  Persistence: Long-term (Memory)


In [8]:
# Demonstrate context assembly - what the LLM actually sees

def show_assembled_context(
    system_prompt: str,
    retrieved_docs: list,
    tool_results: list,
    conversation_history: list,
    implicit_data: dict
):
    """Visualize how context is assembled for an LLM call."""
    
    print("ASSEMBLED CONTEXT FOR LLM")
    print("=" * 60)
    
    # Layer 1: System Prompt
    print("\n[SYSTEM PROMPT]")
    print(f"  {system_prompt}")
    
    # Layer 5: Implicit Data (often injected into system prompt)
    if implicit_data:
        print("\n[IMPLICIT DATA - Injected into instructions]")
        for key, value in implicit_data.items():
            print(f"  {key}: {value}")
    
    # Layer 2: Retrieved Documents
    if retrieved_docs:
        print("\n[RETRIEVED DOCUMENTS]")
        for doc in retrieved_docs:
            print(f"  - {doc}")
    
    # Layer 4: Conversation History
    if conversation_history:
        print("\n[CONVERSATION HISTORY]")
        for msg in conversation_history:
            print(f"  {msg['role']}: {msg['content']}")
    
    # Layer 3: Tool Results
    if tool_results:
        print("\n[TOOL RESULTS]")
        for result in tool_results:
            print(f"  - {result}")
    
    print("\n" + "=" * 60)
    print("→ This complete context is sent to the LLM for each response")

# Example: Well-engineered context
show_assembled_context(
    system_prompt="You are a travel assistant helping {user_name} plan their trip.",
    retrieved_docs=["User prefers window seats", "User is vegetarian"],
    tool_results=["Flight UA123 available: $450", "Weather in Tokyo: 72°F"],
    conversation_history=[
        {"role": "User", "content": "I want to go to Tokyo"},
        {"role": "Agent", "content": "Great choice! When are you planning to travel?"},
        {"role": "User", "content": "Next month"}
    ],
    implicit_data={"user_name": "Alex", "timezone": "PST", "loyalty_tier": "Gold"}
)

ASSEMBLED CONTEXT FOR LLM

[SYSTEM PROMPT]
  You are a travel assistant helping {user_name} plan their trip.

[IMPLICIT DATA - Injected into instructions]
  user_name: Alex
  timezone: PST
  loyalty_tier: Gold

[RETRIEVED DOCUMENTS]
  - User prefers window seats
  - User is vegetarian

[CONVERSATION HISTORY]
  User: I want to go to Tokyo
  Agent: Great choice! When are you planning to travel?
  User: Next month

[TOOL RESULTS]
  - Flight UA123 available: $450
  - Weather in Tokyo: 72°F

→ This complete context is sent to the LLM for each response


### Key Insight: Context Assembly

The core principle of context engineering is that **even advanced models underperform when provided with a limited or poorly constructed view of their operational environment**.

Think of it this way: if an LLM is a brilliant consultant, context engineering is deciding what files to put on their desk before each meeting.

---
## Part 3: Memory Taxonomy

Understanding memory types from cognitive science helps you design better agents. Just like humans, agents benefit from distinct memory systems optimized for different purposes.

In [9]:
# Memory taxonomy based on cognitive science

memory_types = {
    "Semantic Memory": {
        "definition": "Facts and concepts - declarative knowledge",
        "examples": [
            "User is vegetarian",
            "User's budget is $200/night",
            "User prefers window seats"
        ],
        "adk_component": "MemoryService (extracted facts)",
        "persistence": "Long-term, cross-session"
    },
    "Episodic Memory": {
        "definition": "Past experiences and events - what happened",
        "examples": [
            "User's trip to Paris was disrupted by a strike",
            "User successfully used the backup hotel tool last time",
            "User changed seat preference after bad experience"
        ],
        "adk_component": "Session Events / MemoryService",
        "persistence": "Long-term, cross-session"
    },
    "Procedural Memory": {
        "definition": "How to perform tasks - behavioral patterns",
        "examples": [
            "Agent instructions/system prompt",
            "Learned workflows and tool usage patterns",
            "Refined response strategies"
        ],
        "adk_component": "Agent instructions (static or dynamic)",
        "persistence": "Configuration-based"
    },
    "Working Memory": {
        "definition": "Information currently being processed",
        "examples": [
            "Current booking step: payment",
            "Selected flights for comparison",
            "Intermediate calculation results"
        ],
        "adk_component": "Session State",
        "persistence": "Session duration"
    },
    "Short-term Memory": {
        "definition": "Recent conversation context",
        "examples": [
            "What the user just said",
            "Agent's recent responses",
            "Tool results from this turn"
        ],
        "adk_component": "Session Events (context window)",
        "persistence": "Limited by context window"
    }
}

print("MEMORY TAXONOMY FOR AI AGENTS")
print("=" * 70)
for memory_type, details in memory_types.items():
    print(f"\n🧠 {memory_type}")
    print(f"   Definition: {details['definition']}")
    print(f"   ADK Component: {details['adk_component']}")
    print(f"   Persistence: {details['persistence']}")
    print(f"   Examples:")
    for ex in details['examples']:
        print(f"      • {ex}")

MEMORY TAXONOMY FOR AI AGENTS

🧠 Semantic Memory
   Definition: Facts and concepts - declarative knowledge
   ADK Component: MemoryService (extracted facts)
   Persistence: Long-term, cross-session
   Examples:
      • User is vegetarian
      • User's budget is $200/night
      • User prefers window seats

🧠 Episodic Memory
   Definition: Past experiences and events - what happened
   ADK Component: Session Events / MemoryService
   Persistence: Long-term, cross-session
   Examples:
      • User's trip to Paris was disrupted by a strike
      • User successfully used the backup hotel tool last time
      • User changed seat preference after bad experience

🧠 Procedural Memory
   Definition: How to perform tasks - behavioral patterns
   ADK Component: Agent instructions (static or dynamic)
   Persistence: Configuration-based
   Examples:
      • Agent instructions/system prompt
      • Learned workflows and tool usage patterns
      • Refined response strategies

🧠 Working Memory
   De

In [10]:
# Mapping memory types to ADK components

print("\nMEMORY TYPE → ADK COMPONENT MAPPING")
print("=" * 60)
print("""
┌─────────────────────┬────────────────────────┬─────────────────┐
│ Memory Type         │ ADK Component          │ Access Pattern  │
├─────────────────────┼────────────────────────┼─────────────────┤
│ Working Memory      │ Session State          │ Key lookup      │
│ Short-term Memory   │ Session Events         │ Sequential      │
│ Semantic Memory     │ MemoryService          │ Semantic search │
│ Episodic Memory     │ MemoryService          │ Semantic search │
│ Procedural Memory   │ Agent Instructions     │ Direct injection│
└─────────────────────┴────────────────────────┴─────────────────┘
""")


MEMORY TYPE → ADK COMPONENT MAPPING

┌─────────────────────┬────────────────────────┬─────────────────┐
│ Memory Type         │ ADK Component          │ Access Pattern  │
├─────────────────────┼────────────────────────┼─────────────────┤
│ Working Memory      │ Session State          │ Key lookup      │
│ Short-term Memory   │ Session Events         │ Sequential      │
│ Semantic Memory     │ MemoryService          │ Semantic search │
│ Episodic Memory     │ MemoryService          │ Semantic search │
│ Procedural Memory   │ Agent Instructions     │ Direct injection│
└─────────────────────┴────────────────────────┴─────────────────┘



---
## Part 4: Manual Context Management (The Hard Way)

One approach is manually passing conversation history with each request. This works but quickly becomes cumbersome.

In [12]:
def manual_stateful_call(history: list, new_message: str) -> tuple[str, list]:
    """Make an API call with manual history management."""
    # Add the new user message to history
    history.append(types.Content(
        role="user",
        parts=[types.Part.from_text(text=new_message)]
    ))
    
    # Send entire history to the model
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=history
    )
    
    # Add assistant response to history
    history.append(types.Content(
        role="model",
        parts=[types.Part.from_text(text=response.text)]
    ))
    
    return response.text, history

In [13]:
# Initialize empty conversation history
conversation_history = []

# First message
response, conversation_history = manual_stateful_call(
    conversation_history, 
    "My name is Alex and I'm a software engineer who loves hiking."
)
print("User: My name is Alex and I'm a software engineer who loves hiking.")
print(f"Agent: {response}\n")

# Second message - now the model has context!
response, conversation_history = manual_stateful_call(
    conversation_history,
    "What's my name and what do I do for work?"
)
print("User: What's my name and what do I do for work?")
print(f"Agent: {response}")

User: My name is Alex and I'm a software engineer who loves hiking.
Agent: Hi Alex, great to meet you!

A software engineer who loves hiking – that's a fantastic combination! It's interesting how often I hear about people in tech finding their balance and inspiration in nature. Perhaps the problem-solving skills from coding translate well to navigating trails, or maybe the clarity of the mountains helps debug complex thoughts!

Do you find your two passions complement each other? What kind of hiking do you enjoy most – long thru-hikes, challenging mountain ascents, or more leisurely nature walks? And from your software engineering perspective, do you use any cool apps or gadgets to enhance your hiking experience?

Welcome! What brings you here today?

User: What's my name and what do I do for work?
Agent: Your name is **Alex** and you are a **software engineer**.
User: What's my name and what do I do for work?
Agent: Your name is **Alex** and you are a **software engineer**.


### 🟡 Better, But...

This works! The agent now remembers our conversation. However, manual history management has problems:

1. **No persistence** - history is lost when the program restarts
2. **No organization** - hard to manage state separate from messages
3. **No scalability** - grows unbounded, eventually exceeds context limits
4. **No structure** - mixing conversation tracking with business logic

This is where Google ADK's Session management shines.

---
## Part 5: Google ADK Sessions

ADK provides structured session management that handles all the complexity for you.

In [14]:
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai.types import Content, Part

# Constants for our app
APP_NAME = "context_demo"
USER_ID = "user_alex"

In [15]:
# Create a simple conversational agent
agent = LlmAgent(
    model=MODEL_ID,
    name="MemoryAgent",
    instruction="""You are a helpful assistant with excellent memory.
    Remember details the user shares and reference them naturally in conversation.
    Be friendly and conversational."""
)

# Create the session service - this manages conversation state
session_service = InMemorySessionService()

print("✅ Agent and SessionService created")

✅ Agent and SessionService created


In [16]:
# Create a new session for our conversation
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID
)

print(f"✅ Session created")
print(f"   ID: {session.id}")
print(f"   App: {session.app_name}")
print(f"   User: {session.user_id}")

✅ Session created
   ID: ee068cce-b783-45f1-b412-e63b06556d86
   App: context_demo
   User: user_alex


In [17]:
# Create a Runner to execute our agent with session management
runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service
)

print("✅ Runner created - bridges agent execution with session management")

✅ Runner created - bridges agent execution with session management


In [18]:
async def chat(user_message: str, session_id: str) -> str:
    """Send a message and get a response, with automatic session management."""
    content = Content(role="user", parts=[Part.from_text(text=user_message)])
    
    response_text = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response():
            response_text = event.content.parts[0].text
    
    return response_text

In [19]:
# Now let's have a multi-turn conversation!

# Turn 1: Introduce ourselves
response = await chat(
    "Hi! My name is Alex and I work as a software engineer. I love hiking on weekends.",
    session.id
)
print("User: Hi! My name is Alex and I work as a software engineer. I love hiking on weekends.")
print(f"Agent: {response}\n")

User: Hi! My name is Alex and I work as a software engineer. I love hiking on weekends.
Agent: Hi Alex! It's great to meet you. Software engineering sounds like a fascinating field, and hiking on weekends is such a fantastic way to unwind and enjoy nature. I bet you've seen some amazing trails!



In [20]:
# Turn 2: Add more context
response = await chat(
    "I'm planning a trip to Japan next month. I'm really excited about it!",
    session.id
)
print("User: I'm planning a trip to Japan next month. I'm really excited about it!")
print(f"Agent: {response}\n")

User: I'm planning a trip to Japan next month. I'm really excited about it!
Agent: Oh, that's wonderful, Alex! Japan is an incredible country. What parts are you most excited to see or experience there? Are you planning to do any hiking while you're there, given your love for it?



In [21]:
# Turn 3: Test the memory - ask about previous information
response = await chat(
    "Given what you know about me, what activities would you recommend in Japan?",
    session.id
)
print("User: Given what you know about me, what activities would you recommend in Japan?")
print(f"Agent: {response}\n")

User: Given what you know about me, what activities would you recommend in Japan?
Agent: That's an excellent question, Alex, especially knowing you love to hike! Japan truly offers a fantastic blend of nature, culture, and modern innovation.

Given your passion for hiking, I'd definitely recommend exploring some of Japan's stunning trails.

1.  **Kumano Kodo Pilgrimage Routes:** These are ancient pilgrimage trails on the Kii Peninsula, a UNESCO World Heritage site. They offer a beautiful blend of spiritual history and lush, forest hiking. You can do sections that range from a few hours to several days, and there are charming guesthouses and hot springs (onsen) along the way – perfect for unwinding after a long day on the trail!
2.  **Mount Takao (near Tokyo):** If you're looking for a relatively easy and accessible day trip from Tokyo, Mount Takao is fantastic. It has several trails, a cable car and chairlift for those who want to skip part of the ascent, and beautiful views of the cit

In [22]:
# Turn 4: Direct memory test
response = await chat(
    "Quick quiz: What's my name, my job, and my hobby?",
    session.id
)
print("User: Quick quiz: What's my name, my job, and my hobby?")
print(f"Agent: {response}")

User: Quick quiz: What's my name, my job, and my hobby?
Agent: Okay, quick quiz time!

Your name is **Alex**, you work as a **software engineer**, and your hobby is **hiking** on weekends.

How did I do? 😊


### 🟢 Success!

The agent remembers everything from our conversation—name, job, hobby, and travel plans. This is the power of proper session management.

**What's happening behind the scenes:**
1. Each message is recorded as an **Event** in the session
2. The Runner retrieves the full session history on each turn
3. The agent processes the new message with complete context
4. The response is saved as another Event

All of this happens automatically through the ADK framework.

---
## Part 6: Examining Session Structure

Let's look at what's actually stored in a session.

In [23]:
# Retrieve the session to see its current state
current_session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id
)

print("SESSION STRUCTURE")
print("=" * 50)
print(f"ID: {current_session.id}")
print(f"App Name: {current_session.app_name}")
print(f"User ID: {current_session.user_id}")
print(f"Number of Events: {len(current_session.events)}")
print(f"State: {current_session.state}")
print(f"Last Updated: {current_session.last_update_time}")

SESSION STRUCTURE
ID: ee068cce-b783-45f1-b412-e63b06556d86
App Name: context_demo
User ID: user_alex
Number of Events: 8
State: {}
Last Updated: 1767364108.140297


In [24]:
# Examine the events in the session
print("\nSESSION EVENTS (Conversation History)")
print("=" * 50)

for i, event in enumerate(current_session.events):
    # Extract text content from the event
    if event.content and event.content.parts:
        text = event.content.parts[0].text[:100]  # First 100 chars
        if len(event.content.parts[0].text) > 100:
            text += "..."
    else:
        text = "[No text content]"
    
    print(f"\nEvent {i + 1}:")
    print(f"  Author: {event.author}")
    print(f"  Content: {text}")


SESSION EVENTS (Conversation History)

Event 1:
  Author: user
  Content: Hi! My name is Alex and I work as a software engineer. I love hiking on weekends.

Event 2:
  Author: MemoryAgent
  Content: Hi Alex! It's great to meet you. Software engineering sounds like a fascinating field, and hiking on...

Event 3:
  Author: user
  Content: I'm planning a trip to Japan next month. I'm really excited about it!

Event 4:
  Author: MemoryAgent
  Content: Oh, that's wonderful, Alex! Japan is an incredible country. What parts are you most excited to see o...

Event 5:
  Author: user
  Content: Given what you know about me, what activities would you recommend in Japan?

Event 6:
  Author: MemoryAgent
  Content: That's an excellent question, Alex, especially knowing you love to hike! Japan truly offers a fantas...

Event 7:
  Author: user
  Content: Quick quiz: What's my name, my job, and my hobby?

Event 8:
  Author: MemoryAgent
  Content: Okay, quick quiz time!

Your name is **Alex**, you work

### Understanding Events

Events are the building blocks of a session's history. Each event represents:
- A user message
- An agent response
- A tool call or result
- A state update

The Runner automatically:
1. Appends new events to the session
2. Provides the full history to the agent on each turn
3. Persists the session for later retrieval

---
## Part 7: Session State - Working Memory

Sessions can also store **state** - structured data that persists across turns. This maps to **Working Memory** in our taxonomy.

In [25]:
# Create a new session with initial state
session_with_state = await session_service.create_session(
    app_name=APP_NAME,
    user_id="user_with_state",
    state={
        "user_tier": "premium",
        "user:name": "Alex",           # user: prefix = cross-session
        "user:preferences": "window seat, vegetarian",
        "current_step": "initial",     # no prefix = session-only
        "interaction_count": 0
    }
)

print("Session created with initial state:")
print(f"   State: {session_with_state.state}")

Session created with initial state:
   State: {'user_tier': 'premium', 'current_step': 'initial', 'interaction_count': 0, 'user:name': 'Alex', 'user:preferences': 'window seat, vegetarian'}


In [26]:
# Create an agent that uses state in its instructions (state injection)
stateful_agent = LlmAgent(
    model=MODEL_ID,
    name="StatefulAgent",
    instruction="""You are a helpful travel assistant.
    
    User name: {user:name}
    User tier: {user_tier}
    Preferences: {user:preferences}
    Current step: {current_step}
    
    Adapt your responses based on the user's tier and preferences.
    Premium users get more detailed responses."""
)

print("✅ Agent created with state-aware instructions")
print("")
print("State injection placeholders:")
print("   {user:name} → Will be replaced with 'Alex'")
print("   {user_tier} → Will be replaced with 'premium'")
print("   {user:preferences} → Will be replaced with 'window seat, vegetarian'")

✅ Agent created with state-aware instructions

State injection placeholders:
   {user:name} → Will be replaced with 'Alex'
   {user_tier} → Will be replaced with 'premium'
   {user:preferences} → Will be replaced with 'window seat, vegetarian'


### State Prefixes: Controlling Scope

ADK uses prefixes to control state scope and persistence:

| Prefix | Scope | Persistence | Example |
|--------|-------|-------------|--------|
| (none) | Current session | With session | `"current_step": "payment"` |
| `user:` | All sessions for user | Long-term | `"user:name": "Alex"` |
| `app:` | All sessions for app | Long-term | `"app:version": "2.0"` |
| `temp:` | Current invocation | Discarded after | `"temp:api_response": {...}` |

### State vs. Events: When to Use Each

| Aspect | Events (History) | State |
|--------|------------------|-------|
| Purpose | Record what happened | Track what we know |
| Structure | Chronological list | Key-value dictionary |
| Growth | Unbounded (needs compaction) | Bounded (you control keys) |
| Access | Sequential reading | Direct key lookup |
| Memory Type | Short-term (context) | Working memory |

**Use State for:**
- User preferences extracted from conversation (semantic memory)
- Accumulated facts (name, location, interests)
- Workflow progress tracking
- Shopping cart contents

---
## Part 8: The Context Engineering Pipeline *(New)*

Let's visualize how all the pieces fit together in a complete context engineering pipeline.

In [27]:
# Visualize the complete context engineering pipeline

print("""
THE CONTEXT ENGINEERING PIPELINE
================================

User Message
    │
    ▼
┌─────────────────────────────────────────────────────────┐
│                  CONTEXT ASSEMBLY                        │
├─────────────────────────────────────────────────────────┤
│  1. System Instructions (procedural memory)             │
│  2. Retrieved Memories (semantic/episodic memory)       │
│  3. Session State (working memory)                      │
│  4. Recent Events (short-term memory)                   │
│  5. Tool Results (dynamic context)                      │
│  6. User Message (immediate input)                      │
└─────────────────────────────────────────────────────────┘
    │
    ▼
   LLM Processing
    │
    ▼
Agent Response + State Updates + Memory Generation
""")


THE CONTEXT ENGINEERING PIPELINE

User Message
    │
    ▼
┌─────────────────────────────────────────────────────────┐
│                  CONTEXT ASSEMBLY                        │
├─────────────────────────────────────────────────────────┤
│  1. System Instructions (procedural memory)             │
│  2. Retrieved Memories (semantic/episodic memory)       │
│  3. Session State (working memory)                      │
│  4. Recent Events (short-term memory)                   │
│  5. Tool Results (dynamic context)                      │
│  6. User Message (immediate input)                      │
└─────────────────────────────────────────────────────────┘
    │
    ▼
   LLM Processing
    │
    ▼
Agent Response + State Updates + Memory Generation



In [28]:
# Map each component to ADK classes

print("ADK COMPONENT MAPPING")
print("=" * 60)
print("""
Pipeline Stage              │ ADK Component
────────────────────────────┼─────────────────────────────────
System Instructions         │ LlmAgent.instruction
Retrieved Memories          │ MemoryService.search_memory()
Session State               │ Session.state + state injection
Recent Events               │ Session.events
Tool Results                │ Tool return values
User Message                │ Runner.run_async(new_message=...)
────────────────────────────┼─────────────────────────────────
Context Assembly            │ Runner (automatic)
LLM Processing              │ LlmAgent + Model
State Updates               │ EventActions.state_delta
Memory Generation           │ MemoryService.add_session_to_memory()
""")

ADK COMPONENT MAPPING

Pipeline Stage              │ ADK Component
────────────────────────────┼─────────────────────────────────
System Instructions         │ LlmAgent.instruction
Retrieved Memories          │ MemoryService.search_memory()
Session State               │ Session.state + state injection
Recent Events               │ Session.events
Tool Results                │ Tool return values
User Message                │ Runner.run_async(new_message=...)
────────────────────────────┼─────────────────────────────────
Context Assembly            │ Runner (automatic)
LLM Processing              │ LlmAgent + Model
State Updates               │ EventActions.state_delta
Memory Generation           │ MemoryService.add_session_to_memory()



---
## Part 9: Session Persistence Options

ADK provides different SessionService implementations for different needs:

In [29]:
# 1. InMemorySessionService - for development and testing
from google.adk.sessions import InMemorySessionService

dev_service = InMemorySessionService()
print("InMemorySessionService:")
print("   ✓ No setup required")
print("   ✓ Fast for development")
print("   ✗ Data lost on restart")
print("   Best for: Testing, prototyping, examples")

InMemorySessionService:
   ✓ No setup required
   ✓ Fast for development
   ✗ Data lost on restart
   Best for: Testing, prototyping, examples


In [30]:
# 2. DatabaseSessionService - for production with your own database
# (Requires database setup - shown for reference)

print("\nDatabaseSessionService:")
print("   ✓ Persistent storage")
print("   ✓ Works with PostgreSQL, MySQL, SQLite")
print("   ✗ Requires database setup")
print("   Best for: Self-managed production deployments")
print("\n   Example:")
print('   from google.adk.sessions import DatabaseSessionService')
print('   db_service = DatabaseSessionService(db_url="sqlite:///sessions.db")')


DatabaseSessionService:
   ✓ Persistent storage
   ✓ Works with PostgreSQL, MySQL, SQLite
   ✗ Requires database setup
   Best for: Self-managed production deployments

   Example:
   from google.adk.sessions import DatabaseSessionService
   db_service = DatabaseSessionService(db_url="sqlite:///sessions.db")


In [31]:
# 3. VertexAiSessionService - for Google Cloud production
# (Requires GCP setup - shown for reference)

print("\nVertexAiSessionService:")
print("   ✓ Fully managed by Google Cloud")
print("   ✓ Scalable and reliable")
print("   ✓ Integrates with Vertex AI features")
print("   ✗ Requires GCP project and setup")
print("   Best for: Production on Google Cloud")
print("\n   Example:")
print('   from google.adk.sessions import VertexAiSessionService')
print('   vertex_service = VertexAiSessionService(project="my-project", location="us-central1")')


VertexAiSessionService:
   ✓ Fully managed by Google Cloud
   ✓ Scalable and reliable
   ✓ Integrates with Vertex AI features
   ✗ Requires GCP project and setup
   Best for: Production on Google Cloud

   Example:
   from google.adk.sessions import VertexAiSessionService
   vertex_service = VertexAiSessionService(project="my-project", location="us-central1")


---
## Summary: The Context Engineering Hierarchy

```
┌─────────────────────────────────────────────────────────────┐
│                        MEMORY                                │
│         Long-term knowledge across all sessions              │
│     Semantic: "User is allergic to shellfish"               │
│     Episodic: "User's Paris trip had transit issues"        │
├─────────────────────────────────────────────────────────────┤
│                        SESSION                               │
│              Single conversation thread                      │
│   ┌─────────────────────────────────────────────────────┐   │
│   │                    STATE                             │   │
│   │        Working memory for this conversation          │   │
│   │    {"selected_restaurant": "Chez Pierre"}           │   │
│   └─────────────────────────────────────────────────────┘   │
│   ┌─────────────────────────────────────────────────────┐   │
│   │                   EVENTS                             │   │
│   │         Short-term memory (history)                  │   │
│   │   User: "Find me a restaurant"                       │   │
│   │   Agent: "I found Chez Pierre..."                    │   │
│   │   User: "Book it for 7pm"                            │   │
│   └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

### Key Takeaways

1. **LLMs are stateless** - they don't remember anything between API calls
2. **Context Engineering** is a systematic discipline with defined layers
3. **Memory types** (semantic, episodic, procedural, working, short-term) serve different purposes
4. **Sessions** provide conversation-level continuity through event history
5. **State** offers structured working memory with scope prefixes
6. **Memory** (covered in future notebooks) enables cross-session knowledge
7. **SessionService** abstracts away storage complexity
8. **State injection** creates dynamic, personalized agent instructions

### What's Next?

In upcoming notebooks, we'll explore:
- **Stateful Conversations**: State management, compaction strategies
- **Memory Generation**: Converting sessions into long-term knowledge
- **Memory Retrieval**: Proactive vs. reactive memory loading
- **Production Patterns**: Building a complete travel agent

In [32]:
# Clean up - delete our test sessions
await session_service.delete_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id
)

await session_service.delete_session(
    app_name=APP_NAME,
    user_id="user_with_state",
    session_id=session_with_state.id
)

print("✅ Sessions cleaned up")
print("\n🎉 Congratulations! You've learned the fundamentals of context engineering.")

✅ Sessions cleaned up

🎉 Congratulations! You've learned the fundamentals of context engineering.


---
## Further Reading

- **Agentic Design Patterns** by Antonio Gulli (Springer, 2025) — Chapter 1: Prompt Chaining, Chapter 8: Memory Management
- [Google ADK Documentation](https://google.github.io/adk-docs/) — Official reference for Sessions, State, and Memory
- [Context Engineering Whitepaper](https://cloud.google.com/blog/products/ai-machine-learning/context-engineering-for-ai-agents) — Google's perspective on the discipline